In [ ]:
import sys
sys.path.append('${TDL_ROOT_DIR}/John/MNIST_Jan14')
from Trainer import Trainer

In [ ]:
Trainer.root = '${TDL_ROOT_DIR}/John/MNIST_Jan14'
Trainer.results_root = '${TDL_ROOT_DIR}/John/MNIST_Jan14'

In [ ]:
import os
os.chdir("${TDL_ROOT_DIR}/John/MNIST_Jan14")
print(os.getcwd())


In [ ]:
import json
from Trainer import Trainer

# Save original method
if not hasattr(Trainer, "_original_all_bootstrap_stats"):
    Trainer._original_all_bootstrap_stats = Trainer.all_bootstrap_stats

def patched_all_bootstrap_stats(*args, **kwargs):
    out = Trainer._original_all_bootstrap_stats(*args, save=False, **kwargs)

    filename = kwargs.get("filename", "all_models")
    dataset = kwargs.get("dataset", "MNIST")
    p_or_mean = "mean"
    alpha = kwargs.get("alpha", 0.05)

    new_path = (
        f"${TDL_ROOT_DIR}/John/MNIST_Jan14/"
        f"betti_data/{filename}_all_stats_{p_or_mean}_alpha{alpha}.json"
    )

    with open(new_path, "w") as f:
        json.dump(out, f, indent=2)

    print(f"✅ Saved bootstrap stats to:\n{new_path}")
    return out

Trainer.all_bootstrap_stats = patched_all_bootstrap_stats


In [ ]:
from Trainer import Trainer

# Only patch once
if not hasattr(Trainer, "_original_init"):
    Trainer._original_init = Trainer.__init__

def _patched_init(self, *args, **kwargs):
    if "root" not in kwargs or kwargs["root"] is None:
        kwargs["root"] = "${TDL_ROOT_DIR}/John/MNIST_Jan14"
    Trainer._original_init(self, *args, **kwargs)

Trainer.__init__ = _patched_init


In [ ]:
t = Trainer(
    dataset="MNIST",
    hidden_dims=[30]*8,
    act_fn=None,
    study_name="test"
)

print("root:", t.root)


In [ ]:
import numpy as np
from Trainer import Trainer

if not hasattr(Trainer, "_original_betti"):
    Trainer._original_betti = Trainer._betti_at_eta_one_dim

def safe_betti(diagram_dim, eta):
    arr = np.asarray(diagram_dim)
    if arr.ndim == 1 and arr.size == 2:
        arr = arr.reshape(1, 2)
    births = arr[:, 0]
    deaths = arr[:, 1]
    return np.sum((births <= eta) & (eta < deaths))

Trainer._betti_at_eta_one_dim = staticmethod(safe_betti)


In [ ]:
import os
import dill
import numpy as np

EXPECTED_LAYERS = 9  # 8 hidden + input

def empty_layer_like(layer):
    # Same number of homology dims, but empty
    return [np.empty((0, 2)) for _ in layer]

for model in os.listdir("."):
    agg = os.path.join(model, "AGG_TSS")
    if not os.path.isdir(agg):
        continue

    for fname in os.listdir(agg):
        if not fname.endswith(".dill"):
            continue

        path = os.path.join(agg, fname)
        with open(path, "rb") as f:
            diagram = dill.load(f)

        if len(diagram) >= EXPECTED_LAYERS:
            continue

        # Pad with empty layers
        while len(diagram) < EXPECTED_LAYERS:
            diagram.append(empty_layer_like(diagram[0]))

        with open(path, "wb") as f:
            dill.dump(diagram, f)

        print(f"✔ padded {model}/AGG_TSS/{fname} to {EXPECTED_LAYERS} layers")


In [ ]:
import dill
with open("512x8_relu/AGG_TSS/model_0.dill", "rb") as f:
    d = dill.load(f)

print("num layers:", len(d))
for i, layer in enumerate(d):
    print(i, len(layer), [arr.shape for arr in layer])


In [ ]:
from Trainer import Trainer

# Save original method once
if not hasattr(Trainer, "_orig_get_betti_mat"):
    Trainer._orig_get_betti_mat = Trainer.get_betti_mat

def set_eta(eta):
    def patched_get_betti_mat(self, dir_name, *args, **kwargs):
        return Trainer._orig_get_betti_mat(
            self,
            dir_name=dir_name,
            eta=eta,
            dims=[0],              # β₀
            use_running_min=False
        )
    Trainer.get_betti_mat = patched_get_betti_mat


In [ ]:
for eta in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    set_eta(eta)
    data = Trainer.all_bootstrap_stats(
        filename=f"class0_eta{eta}",
        studies=["512x8_relu", "512x8_leaky","512x8_tanh"],
        dataset="MNIST",
        dir_name="ripser_class_0_norm"
    )
    Trainer.get_tsc(
        data,
        to_calculate=[r'^512x8_'],
        metric="p_gt0",
        title=rf"Within-class TSS (digit 0, $\eta={eta}$)",
        legend=["512x8_relu", "512x8_leaky","512x8_tanh"],
        plot=True
    )


In [ ]:
import jax
import numpy as np
from Trainer import Trainer

etas = [0.1, 0.2, 0.3]
root = "${TDL_ROOT_DIR}/John/MNIST_Jan14"

for eta in etas:
    print("\n" + "="*60)
    print(f"=== Within-class TSS (digit 0), eta = {eta} ===")

    # ---- STEP A: fix eta ----
    set_eta(eta)
    print(f"[DEBUG] Using eta = {eta}")

    # ---- STEP B: inspect raw Betti matrices ----
    betti_mats = {}

    for arch, act in [
        ("512x8_relu", jax.nn.relu),
        ("512x8_leaky", jax.nn.leaky_relu),
        ("512x8_tanh", jax.nn.tanh),
    ]:
        trainer_dbg = Trainer(
            dataset="MNIST",
            hidden_dims=[512]*8,
            act_fn=act,
            study_name=arch,
            root=root
        )

        bm = trainer_dbg.get_betti_mat("ripser_class_0_norm")
        betti_mats[arch] = bm

        print(f"\n[DEBUG] {arch}")
        print("  shape:", bm.shape)
        print("  mean per layer:", np.round(bm.mean(axis=0), 2))
        print("  std  per layer:", np.round(bm.std(axis=0), 2))

    # ---- STEP C: run bootstrap ----
    data = Trainer.all_bootstrap_stats(
        filename=f"class0_eta{eta}",
        studies=["512x8_relu", "512x8_leaky", "512x8_tanh"],
        dataset="MNIST",
        dir_name="ripser_class_0_norm"
    )

    print("\n[DEBUG] Bootstrap keys:", data.keys())

    # ---- STEP D: inspect bootstrap layer stats ----
    print("\n[DEBUG] Bootstrap stats per layer (pairwise):")
    for A in data:
        for B in data[A]:
            print(f"\n  {A} vs {B}")
            for ell in range(1, 6):  # first few hidden layers
                node = data[A][B][ell]
                print(
                    f"    layer {ell}: "
                    f"mean={node['mean']:.3f}, "
                    f"se={node['se']:.3f}, "
                    f"p_gt0={node['p_gt0']:.3f}, "
                    f"p_lt0={node['p_lt0']:.3f}, "
                    f"p_eq0={node['p_eq0']:.3f}"
                )

    # ---- STEP E: inspect TSS numerically ----
    tss = Trainer.get_tsc(
        data,
        to_calculate=[r'^512x8_'],
        metric="p_gt0",
        plot=False
    )

    print("\n[DEBUG] Final TSS values:")
    for k, v in tss.items():
        print(f"  {k}: {np.round(v, 3)}")

    # ---- STEP F: plot ----
    Trainer.get_tsc(
        data,
        to_calculate=[r'^512x8_'],
        metric="p_gt0",
        title=rf"Within-class TSS (digit 0, $\eta={eta}$)",
        legend=["ReLU", "Leaky", "Tanh"],
        legend_title="Activation",
        plot=True
    )
